# Notebook 02: Data Cleaning, Quality Auditing, and Plausibility Validation
## BDA Mini-Project: Air Pollution Analysis Across India

### Objectives:
- Inspect missing values, duplicate records, and invalid negative measurements.
- Establish physical boundary constraints for environmental sensor readings.
- Execute PySpark data cleaning and audit before-and-after data retention.
- Generate a reproducible data quality report.

In [1]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import src.config as config
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, min, max

spark = (
    SparkSession.builder
    .appName('BDA_02_Data_Cleaning')
    .master('local[2]')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.ui.enabled', 'false')
    .getOrCreate()
)
print('SparkSession initialized successfully.')

SparkSession initialized successfully.


### 1. Ingesting Raw Data Sample & Auditing Missing / Invalid Values

In [2]:
sample_path = PROJECT_ROOT / 'data' / 'sample' / 'measurements_2024_01.parquet'
df_raw = spark.read.parquet(str(sample_path))
raw_total = df_raw.count()
print(f'Raw Observation Count in January 2024: {raw_total:,}')

# Audit nulls and negative values
audit_df = df_raw.select([
    count(when(col(c).isNull(), c)).alias(f'{c}_nulls')
    for c in ['station_id', 'parameter_name', 'collected_at', 'value']
] + [
    count(when(col('value') < 0, 1)).alias('negative_readings'),
    count(when(col('value') == 0, 1)).alias('zero_readings'),
])
audit_df.show()

Raw Observation Count in January 2024: 4,002,852


+----------------+--------------------+------------------+-----------+-----------------+-------------+
|station_id_nulls|parameter_name_nulls|collected_at_nulls|value_nulls|negative_readings|zero_readings|
+----------------+--------------------+------------------+-----------+-----------------+-------------+
|               0|                   0|                 0|          0|                0|        44851|
+----------------+--------------------+------------------+-----------+-----------------+-------------+



### 2. Physical Boundary Plausibility Filters

In [3]:
from src.clean_data import clean_measurements_df, compute_quality_summary

df_clean = clean_measurements_df(df_raw)
clean_total = df_clean.count()
print(f'Clean Valid Observations: {clean_total:,}')
print(f'Filtered Invalid / Extreme Spikes: {raw_total - clean_total:,}')
print(f'Data Retention Rate: {(clean_total / raw_total * 100):.2f}%')

Clean Valid Observations: 4,002,852
Filtered Invalid / Extreme Spikes: 0
Data Retention Rate: 100.00%


### 3. Before-and-After Quality Comparison Summary

In [4]:
quality_json_path = PROJECT_ROOT / 'outputs' / 'metrics' / 'data_quality_summary.json'
if quality_json_path.exists():
    with open(quality_json_path) as f:
        q_metrics = json.load(f)
    display_df = pd.DataFrame([q_metrics])
    print(display_df.to_string(index=False))
else:
    print('Quality summary loaded from execution.')

 raw_record_count  clean_record_count  filtered_record_count  retention_percentage  ingest_time_seconds  cleaning_time_seconds  save_time_seconds
         46937170            46937146                     24                 100.0                12.53                 170.43             166.93


### 4. Cleanup

In [5]:
spark.stop()
print('Spark session closed cleanly.')

Spark session closed cleanly.
